# Parcel ISPC — Condition Contrasts (Left-Wing Subjects)

Compare ISPC values across conditions for parcels that are **FDR-significant and ISPC > 0**
in at least one condition (from `parcel_ispc_leftwing.ipynb` permutation test).

**Three contrasts:**
1. Agreed (AntiRight + ProLeft) vs Disagreed (AntiLeft + ProRight)
2. Within Agreed: AntiRight vs ProLeft
3. Within Disagreed: AntiLeft vs ProRight

**Two levels:**
- **Section A — Subject level**: paired t-test on LOO-ISPC per subject, FDR-corrected across parcels
- **Section B — Group mean**: descriptive comparison of group mean ISPC across conditions

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import fdrcorrection
import sys
from IPython.display import display

## Configuration

In [ ]:
ROOT         = Path('/path/to/project')  # CHANGE THIS to your local path
OUTPUT_DIR   = ROOT / 'data/derivatives/parcel_ispc/leftwing'
CONTRAST_DIR = OUTPUT_DIR / 'contrasts'
CONTRAST_DIR.mkdir(parents=True, exist_ok=True)

ATLAS_NII  = ROOT / 'data/atlases/Schaefer2018_tf_2mm_400Parcels7Networks_plus_TianS3.dseg.nii.gz'
LABELS_TSV = ROOT / 'data/atlases/Schaefer2018_400Parcels7Networks_plus_TianS3_labels.tsv'

sys.path.insert(0, str(ROOT / 'yy-fMRI-kit/src'))
from yy_fmri_kit.event_isc.contrast import parcels_to_nifti

RUN_TYPES      = ['AntiLeft', 'AntiRight', 'ProLeft', 'ProRight']
AGREED         = ['AntiRight', 'ProLeft']
DISAGREED      = ['AntiLeft',  'ProRight']
FDR_Q          = 0.05
ISPC_ABS_THRESH = 0.1   # parcel included if |isc_mean| >= this in >=1 condition of the contrast

ANOVA_DIR = CONTRAST_DIR / 'anova'
ANOVA_DIR.mkdir(parents=True, exist_ok=True)

## Load Data

Loads the significance table and per-subject ISPC matrices saved by `parcel_ispc_leftwing.ipynb`.

In [ ]:
# Significance table (one row per condition x parcel)
sig_df = pd.read_csv(OUTPUT_DIR / 'parcel_isc_B_significance.csv')
print(f'Significance table: {sig_df.shape}  conditions: {sig_df.condition.unique().tolist()}')

# Per-subject ISPC matrices (rows=subjects, cols=parcels)
isc_subj = {}
isc_mean = {}

for cond in RUN_TYPES:
    csv_path = OUTPUT_DIR / f'parcel_isc_B_{cond}_persubject.csv'
    df = pd.read_csv(csv_path, index_col=0)
    isc_subj[cond] = df
    print(f'  {cond}: {df.shape[0]} subjects x {df.shape[1]} parcels')

# Parcel name list
ALL_PARCEL_NAMES = list(isc_subj[RUN_TYPES[0]].columns)

# Align to common subjects
common_subjects = sorted(
    set(isc_subj[RUN_TYPES[0]].index)
    .intersection(*[set(isc_subj[c].index) for c in RUN_TYPES[1:]])
)
print(f'\nCommon subjects: {len(common_subjects)}')

# Re-index and convert to numpy
subj_arr = {}   # dict[cond] -> np.ndarray (n_subjects, n_parcels)
mean_arr = {}   # dict[cond] -> np.ndarray (n_parcels,)
for cond in RUN_TYPES:
    arr = isc_subj[cond].loc[common_subjects, ALL_PARCEL_NAMES].to_numpy(dtype=np.float64)
    subj_arr[cond] = arr
    mean_arr[cond] = np.nanmean(arr, axis=0)

## Parcel Selection

Parcel selection is **per contrast**, not global. For each contrast, include parcels that satisfy
both criteria in the conditions entering that contrast:

1. **FDR-significant** (`p_fdr < 0.05`) in at least one of the contrast's conditions
2. **|ISPC| ≥ 0.1** in at least one of those same conditions

A parcel only needs to pass either criterion in *any one* condition — it does not need to be
significant in both conditions simultaneously.

In [ ]:
# Overview: how many parcels are FDR-significant per condition
for cond in RUN_TYPES:
    n_sig = (sig_df[(sig_df['condition'] == cond) & (sig_df['significant'] == True)]).shape[0]
    n_thr = (sig_df[(sig_df['condition'] == cond) & (sig_df['isc_mean'].abs() >= ISPC_ABS_THRESH)]).shape[0]
    print(f'{cond}: {n_sig} FDR-sig parcels | {n_thr} parcels with |ISPC| >= {ISPC_ABS_THRESH}')

print('\nPer-contrast parcel counts are shown when each contrast is run below.')

## Helper Functions

In [ ]:
def get_parcels_for_contrast(conditions):
    """
    Select parcels that are:
      (1) FDR-significant in >=1 of the given conditions, AND
      (2) |isc_mean| >= ISPC_ABS_THRESH in >=1 of those same conditions.
    """
    cond_mask = sig_df['condition'].isin(conditions)
    sig_set   = set(sig_df.loc[cond_mask & (sig_df['significant'] == True), 'parcel_name'])
    thr_set   = set(sig_df.loc[cond_mask & (sig_df['isc_mean'].abs() >= ISPC_ABS_THRESH), 'parcel_name'])
    return sorted(sig_set & thr_set)


def run_contrast(a_arr, b_arr, label_a, label_b, parcel_list):
    """Paired t-test for each parcel in parcel_list; returns DataFrame sorted by p."""
    rows = []
    for pname in parcel_list:
        idx  = ALL_PARCEL_NAMES.index(pname)
        a, b = a_arr[:, idx], b_arr[:, idx]
        mask = ~(np.isnan(a) | np.isnan(b))
        a_c, b_c = a[mask], b[mask]
        if len(a_c) < 3:
            continue
        t, p      = stats.ttest_rel(a_c, b_c)
        diff      = a_c - b_c
        mean_diff = float(np.mean(diff))
        cohen_d   = mean_diff / (float(np.std(diff, ddof=1)) + 1e-12)
        rows.append({
            'parcel_name'     : pname,
            f'mean_{label_a}' : round(float(np.mean(a_c)), 4),
            f'mean_{label_b}' : round(float(np.mean(b_c)), 4),
            'mean_diff (A-B)' : round(mean_diff, 4),
            'cohen_d'         : round(cohen_d, 3),
            't'               : round(float(t), 3),
            'p_raw'           : round(float(p), 5),
            'n_subjects'      : int(mask.sum()),
        })
    df = pd.DataFrame(rows)
    if len(df) > 0:
        _, p_fdr = fdrcorrection(df['p_raw'].values, alpha=FDR_Q)
        df['p_fdr']       = np.round(p_fdr, 5)
        df['significant'] = df['p_fdr'] < FDR_Q
        df = df.sort_values('p_raw').reset_index(drop=True)
    return df


def box_plot(result_df, a_arr, b_arr, label_a, label_b,
             color_a='#5B8DB8', color_b='#6AA96B', title='', out_path=None):
    """Box plot with paired subject dots for FDR-sig parcels (top 5 fallback)."""
    plot_df = result_df[result_df['significant']]
    if len(plot_df) == 0:
        plot_df = result_df.head(5)
        print('  No FDR-significant parcels — showing top 5 by p-value')

    n   = len(plot_df)
    fig, axes = plt.subplots(1, n, figsize=(max(5, n * 3.0), 4.5), squeeze=False)
    rng = np.random.default_rng(42)

    for i, (_, row) in enumerate(plot_df.iterrows()):
        ax     = axes[0, i]
        idx    = ALL_PARCEL_NAMES.index(row['parcel_name'])
        a_vals = a_arr[:, idx]
        b_vals = b_arr[:, idx]

        # Box plots
        bp = ax.boxplot(
            [a_vals, b_vals],
            positions=[0, 1], widths=0.45, patch_artist=True,
            medianprops=dict(color='black', linewidth=2),
            boxprops=dict(linewidth=1.2),
            whiskerprops=dict(linewidth=1.0),
            capprops=dict(linewidth=1.0),
            flierprops=dict(marker=''),
        )
        for patch, col in zip(bp['boxes'], [color_a, color_b]):
            patch.set_facecolor(col)
            patch.set_alpha(0.65)

        # Individual subject dots
        for xi, vals in enumerate([a_vals, b_vals]):
            jitter = rng.uniform(-0.08, 0.08, size=len(vals))
            ax.scatter(xi + jitter, vals, color='black', s=16, alpha=0.55, zorder=3)

        # Paired lines
        for av, bv in zip(a_vals, b_vals):
            ax.plot([0, 1], [av, bv], color='grey', alpha=0.20, linewidth=0.8)

        # Significance bracket
        p_fdr = row['p_fdr']
        star  = '***' if p_fdr < 0.001 else '**' if p_fdr < 0.01 else '*' if p_fdr < 0.05 else 'n.s.'
        y_top = max(np.nanmax(a_vals), np.nanmax(b_vals)) + 0.03
        ax.plot([0, 0, 1, 1], [y_top, y_top+0.01, y_top+0.01, y_top], color='black', linewidth=1)
        ax.text(0.5, y_top + 0.012, star, ha='center', va='bottom', fontsize=10)

        # Title
        parts = row['parcel_name'].split('_')
        short = ('_'.join(parts[-4:-2]) + '\n' + '_'.join(parts[-2:])) \
                if len(parts) >= 4 else row['parcel_name']
        ax.set_title(short, fontsize=8, pad=3, linespacing=1.3)
        ax.set_xticks([0, 1])
        ax.set_xticklabels([label_a, label_b], fontsize=8)
        ax.set_ylabel('LOO-ISPC', fontsize=8)
        ax.tick_params(labelsize=7)
        ax.text(0.97, 0.03,
                f'p_fdr={p_fdr:.3f}\nd={row["cohen_d"]:.2f}',
                transform=ax.transAxes, ha='right', va='bottom', fontsize=7)

    fig.suptitle(title, fontsize=11, y=1.02)
    plt.tight_layout()
    if out_path:
        plt.savefig(str(out_path), dpi=150, bbox_inches='tight')
    plt.show()
    return fig

def get_parcels_for_anova():
    """FDR-sig AND |ISPC|>=0.1 in any of the 4 conditions — same as Contrast 1 criterion."""
    return get_parcels_for_contrast(['AntiLeft', 'AntiRight', 'ProLeft', 'ProRight'])



def run_anova_parcelwise(subj_arr, parcel_list, fdr_q=FDR_Q):
    """
    Vectorized 2x2 within-subjects RM-ANOVA per parcel.

    Factors
    -------
    Agreement : Agreed (ProLeft, AntiRight) vs Disagreed (ProRight, AntiLeft)
    Group     : Ingroup/Left (ProLeft, AntiLeft) vs Outgroup/Right (ProRight, AntiRight)

    Cell mapping
    ------------
    Agreed   x Ingroup  = ProLeft
    Agreed   x Outgroup = AntiRight
    Disagreed x Ingroup  = AntiLeft
    Disagreed x Outgroup = ProRight

    The Agreement x Group interaction contrast equals (ProLeft + ProRight) - (AntiRight + AntiLeft),
    which is the Pro-vs-Anti (Valence) contrast — so Pro/Anti emerges from the interaction.

    For a balanced 2x2 within-subjects design each effect has df1=1, so F = t^2
    exactly. Sphericity is trivially satisfied when df1=1.
    FDR correction (BH, q=fdr_q) applied independently per effect.
    """
    PL = subj_arr['ProLeft']
    AR = subj_arr['AntiRight']
    PR = subj_arr['ProRight']
    AL = subj_arr['AntiLeft']
    n  = PL.shape[0]

    idxs = [ALL_PARCEL_NAMES.index(p) for p in parcel_list]
    PL, AR, PR, AL = PL[:, idxs], AR[:, idxs], PR[:, idxs], AL[:, idxs]

    # Factor 1 marginals (Agreement)
    agreed    = (PL + AR) / 2
    disagreed = (PR + AL) / 2

    # Factor 2 marginals (Group: Ingroup vs Outgroup)
    ingroup  = (PL + AL) / 2   # ProLeft + AntiLeft (content about Left / ingroup)
    outgroup = (AR + PR) / 2   # AntiRight + ProRight (content about Right / outgroup)

    # Interaction contrast: (Agreed_Ingroup - Agreed_Outgroup) - (Disagreed_Ingroup - Disagreed_Outgroup)
    # = (PL - AR) - (AL - PR) = (PL + PR) - (AR + AL)  [= Pro - Anti, i.e. Valence]
    # Positive = Pro > Anti ISPC (within this Agreement x Group structure)
    int_c = (PL + PR) - (AR + AL)

    # Vectorized tests across all parcels simultaneously
    t_a, p_a = stats.ttest_rel(agreed,  disagreed, axis=0)
    t_g, p_g = stats.ttest_rel(ingroup, outgroup,  axis=0)
    t_i, p_i = stats.ttest_1samp(int_c, 0,         axis=0)

    df2 = n - 1
    F_a, F_g, F_i = t_a**2, t_g**2, t_i**2

    # Signed Cohen's dz (paired within-subjects effect size)
    diff_a = agreed - disagreed
    diff_g = ingroup - outgroup
    dz_a   = diff_a.mean(0) / (diff_a.std(0, ddof=1) + 1e-12)
    dz_g   = diff_g.mean(0) / (diff_g.std(0, ddof=1) + 1e-12)
    dz_i   = int_c.mean(0)  / (int_c.std(0,  ddof=1) + 1e-12)

    # Partial eta-squared: F / (F + df2) when df1=1
    pet_a = F_a / (F_a + df2)
    pet_g = F_g / (F_g + df2)
    pet_i = F_i / (F_i + df2)

    # Independent FDR corrections, one per effect
    _, p_a_fdr = fdrcorrection(p_a, alpha=fdr_q)
    _, p_g_fdr = fdrcorrection(p_g, alpha=fdr_q)
    _, p_i_fdr = fdrcorrection(p_i, alpha=fdr_q)

    rows = []
    for k, pname in enumerate(parcel_list):
        rows.append({
            'parcel_name':          pname,
            'mean_ProLeft':         round(float(PL[:, k].mean()), 5),
            'mean_AntiRight':       round(float(AR[:, k].mean()), 5),
            'mean_ProRight':        round(float(PR[:, k].mean()), 5),
            'mean_AntiLeft':        round(float(AL[:, k].mean()), 5),
            # Agreement main effect
            'F_agreement':          round(float(F_a[k]), 4),
            'p_agreement':          round(float(p_a[k]), 6),
            'p_fdr_agreement':      round(float(p_a_fdr[k]), 6),
            'dz_agreement':         round(float(dz_a[k]), 4),
            'pet_agreement':        round(float(pet_a[k]), 4),
            'sig_agreement':        bool(p_a_fdr[k] < fdr_q),
            # Group main effect (Ingroup vs Outgroup)
            'F_group':              round(float(F_g[k]), 4),
            'p_group':              round(float(p_g[k]), 6),
            'p_fdr_group':          round(float(p_g_fdr[k]), 6),
            'dz_group':             round(float(dz_g[k]), 4),
            'pet_group':            round(float(pet_g[k]), 4),
            'sig_group':            bool(p_g_fdr[k] < fdr_q),
            # Agreement x Group interaction (= Pro vs Anti / Valence)
            'F_interaction':        round(float(F_i[k]), 4),
            'p_interaction':        round(float(p_i[k]), 6),
            'p_fdr_interaction':    round(float(p_i_fdr[k]), 6),
            'dz_interaction':       round(float(dz_i[k]), 4),
            'pet_interaction':      round(float(pet_i[k]), 4),
            'sig_interaction':      bool(p_i_fdr[k] < fdr_q),
            'n_subjects':           n,
        })

    df = pd.DataFrame(rows)
    print(f'2x2 RM-ANOVA complete: {len(df)} parcels tested')
    print(f'  Agreement:   {df["sig_agreement"].sum()} FDR-sig')
    print(f'  Group:       {df["sig_group"].sum()} FDR-sig')
    print(f'  Interaction: {df["sig_interaction"].sum()} FDR-sig')
    return df
def make_anova_view_df(anova_df, effect):
    """
    Reformat anova_df for one ANOVA effect into the result_df format expected
    by _all_arr() and _sig_mask_arr() — columns: parcel_name, cohen_d, significant.
    """
    return anova_df[['parcel_name', f'dz_{effect}', f'sig_{effect}']].rename(columns={
        f'dz_{effect}':  'cohen_d',
        f'sig_{effect}': 'significant',
    }).copy()


def box_plot_anova(anova_df, effect, subj_arr, all_parcel_names, title='', out_path=None,
                   n_parcels=6, color_agreed='#5B8DB8', color_disagreed='#E07B54',
                   mark_other_effects=False):
    """
    2x2 interaction-style plot for FDR-sig parcels of a given ANOVA effect.
    x-axis = Group (Ingroup / Outgroup); two lines coloured by Agreement.
    If mark_other_effects=True, adds APA-style significance brackets for
    Agreement and Group main effects in each parcel panel.
    Parcels are sorted by p_fdr of the requested effect (top n_parcels shown).
    """
    def _stars(p):
        return '***' if p < 0.001 else '**' if p < 0.01 else '*'

    sig_col = f'sig_{effect}'
    plot_df = anova_df[anova_df[sig_col]].sort_values(f'p_fdr_{effect}').head(n_parcels).copy()
    if len(plot_df) == 0:
        plot_df = anova_df.sort_values(f'p_{effect}').head(min(n_parcels, 5)).copy()
        print(f'  No FDR-sig parcels for {effect} — showing top {min(n_parcels,5)} by p-value')

    n_panels = len(plot_df)
    fig, axes = plt.subplots(1, n_panels, figsize=(n_panels * 3.2, 5.0), squeeze=False)

    for i, (_, row) in enumerate(plot_df.iterrows()):
        ax  = axes[0, i]
        idx = all_parcel_names.index(row['parcel_name'])
        PL  = subj_arr['ProLeft'][:,  idx]   # Agreed   x Ingroup
        AR  = subj_arr['AntiRight'][:, idx]  # Agreed   x Outgroup
        AL  = subj_arr['AntiLeft'][:, idx]   # Disagreed x Ingroup
        PR  = subj_arr['ProRight'][:, idx]   # Disagreed x Outgroup

        # Group means
        ax.plot([0, 1], [PL.mean(), AR.mean()], 'o-', color=color_agreed,
                linewidth=2.5, label='Agreed', zorder=3)
        ax.plot([0, 1], [AL.mean(), PR.mean()], 'o-', color=color_disagreed,
                linewidth=2.5, label='Disagreed', zorder=3)

        # Subject-level lines
        for pl, ar in zip(PL, AR):
            ax.plot([0, 1], [pl, ar], color=color_agreed,    alpha=0.12, linewidth=0.6)
        for al, pr in zip(AL, PR):
            ax.plot([0, 1], [al, pr], color=color_disagreed, alpha=0.12, linewidth=0.6)

        ax.set_xticks([0, 1])
        ax.set_xticklabels(['Ingroup', 'Outgroup'], fontsize=9)
        ax.set_ylabel('LOO-ISPC', fontsize=8)
        ax.tick_params(labelsize=7)

        # Title: parcel name + this effect's stats
        p_fdr = row[f'p_fdr_{effect}']
        dz    = row[f'dz_{effect}']
        parts = row['parcel_name'].split('_')
        short = ('_'.join(parts[-4:-2]) + '\n' + '_'.join(parts[-2:])) if len(parts) >= 4 else row['parcel_name']
        ax.set_title(f'{short}\n{_stars(p_fdr)} p_fdr={p_fdr:.3f}  dz={dz:.2f}',
                     fontsize=7, pad=4)
        ax.legend(fontsize=7, loc='upper right')

        if mark_other_effects:
            y_lo, y_hi = ax.get_ylim()
            y_rng = y_hi - y_lo

            # ── Agreement main effect: vertical bracket on right ──────────
            if row.get('sig_agreement', False):
                y_ag = (PL.mean() + AR.mean()) / 2   # Agreed line midpoint
                y_dg = (AL.mean() + PR.mean()) / 2   # Disagreed line midpoint
                s    = _stars(row['p_fdr_agreement'])
                xb   = 1.22
                xt   = y_rng * 0.025                 # tick length in y-units
                # bracket: two ticks + spine
                ax.plot([xb, xb], [y_ag, y_dg],
                        color='#333333', lw=0.9, clip_on=False)
                ax.plot([xb - 0.06, xb], [y_ag, y_ag],
                        color='#333333', lw=0.9, clip_on=False)
                ax.plot([xb - 0.06, xb], [y_dg, y_dg],
                        color='#333333', lw=0.9, clip_on=False)
                ax.text(xb + 0.05, (y_ag + y_dg) / 2,
                        f'Agree\n{s}',
                        ha='left', va='center', fontsize=6, clip_on=False,
                        color='#333333')

            # ── Group main effect: horizontal bracket above plot ──────────
            if row.get('sig_group', False):
                s   = _stars(row['p_fdr_group'])
                yb  = y_hi + y_rng * 0.10
                yt  = y_rng * 0.025
                ax.plot([0, 0, 1, 1], [yb, yb + yt, yb + yt, yb],
                        color='#333333', lw=0.9, clip_on=False)
                ax.text(0.5, yb + yt * 1.8, f'Group: {s}',
                        ha='center', va='bottom', fontsize=6, clip_on=False,
                        color='#333333')

    fig.suptitle(title, fontsize=11, y=1.04)
    plt.tight_layout()
    if out_path:
        plt.savefig(str(out_path), dpi=150, bbox_inches='tight')
    plt.show()
    return fig

---
## Section A — Subject-Level Paired t-Tests

One LOO-ISPC value per subject per condition; paired t-test across subjects.
FDR correction (BH, q = 0.05) applied across all selected parcels within each contrast.

### Contrast 1 — Agreed vs Disagreed

**Agreed** = (AntiRight + ProLeft) / 2 per subject  
**Disagreed** = (AntiLeft + ProRight) / 2 per subject  
Positive `mean_diff` = higher ISPC for agreed content.

In [ ]:
# Parcels: FDR-sig AND |ISPC| >= 0.1 in any of the 4 conditions
parcels_c1 = get_parcels_for_contrast(['AntiLeft', 'AntiRight', 'ProLeft', 'ProRight'])
print(f'Parcels selected for Contrast 1: {len(parcels_c1)}')

agreed_subj    = (subj_arr['AntiRight'] + subj_arr['ProLeft'])  / 2
disagreed_subj = (subj_arr['AntiLeft']  + subj_arr['ProRight']) / 2

c1 = run_contrast(agreed_subj, disagreed_subj, 'Agreed', 'Disagreed', parcels_c1)
print(f'Contrast 1 — Agreed vs Disagreed  |  {len(c1)} parcels tested  |  {c1["significant"].sum()} FDR-sig')
display(c1.head(15))

In [ ]:
box_plot(
    c1, agreed_subj, disagreed_subj, 'Agreed', 'Disagreed',
    color_a='#5B8DB8', color_b='#E07B54',
    title='Contrast 1: Agreed vs Disagreed — Subject-Level LOO-ISPC',
    out_path=CONTRAST_DIR / 'subj_contrast1_agreed_vs_disagreed.png',
)

### Contrast 2 — AntiRight vs ProLeft (within Agreed)

Tests whether the two agreed sub-types differ in neural synchrony.

In [ ]:
# Parcels: FDR-sig AND |ISPC| >= 0.1 in AntiRight or ProLeft
parcels_c2 = get_parcels_for_contrast(['AntiRight', 'ProLeft'])
print(f'Parcels selected for Contrast 2: {len(parcels_c2)}')

c2 = run_contrast(subj_arr['AntiRight'], subj_arr['ProLeft'], 'AntiRight', 'ProLeft', parcels_c2)
print(f'Contrast 2 — AntiRight vs ProLeft  |  {len(c2)} parcels tested  |  {c2["significant"].sum()} FDR-sig')
display(c2.head(15))

In [ ]:
box_plot(
    c2, subj_arr['AntiRight'], subj_arr['ProLeft'], 'AntiRight', 'ProLeft',
    color_a='#5B8DB8', color_b='#6AA96B',
    title='Contrast 2: AntiRight vs ProLeft — Subject-Level LOO-ISPC',
    out_path=CONTRAST_DIR / 'subj_contrast2_antiright_vs_proleft.png',
)

### Contrast 3 — AntiLeft vs ProRight (within Disagreed)

Tests whether the two disagreed sub-types differ in neural synchrony.

In [ ]:
# Parcels: FDR-sig AND |ISPC| >= 0.1 in AntiLeft or ProRight
parcels_c3 = get_parcels_for_contrast(['AntiLeft', 'ProRight'])
print(f'Parcels selected for Contrast 3: {len(parcels_c3)}')

c3 = run_contrast(subj_arr['AntiLeft'], subj_arr['ProRight'], 'AntiLeft', 'ProRight', parcels_c3)
print(f'Contrast 3 — AntiLeft vs ProRight  |  {len(c3)} parcels tested  |  {c3["significant"].sum()} FDR-sig')
display(c3.head(15))

In [ ]:
box_plot(
    c3, subj_arr['AntiLeft'], subj_arr['ProRight'], 'AntiLeft', 'ProRight',
    color_a='#E07B54', color_b='#9B6BB5',
    title='Contrast 3: AntiLeft vs ProRight — Subject-Level LOO-ISPC',
    out_path=CONTRAST_DIR / 'subj_contrast3_antileft_vs_proright.png',
)

---
## Brain Maps — FDR-Significant Parcels per Contrast

Yabplot cortical surface maps and subcortical volume slices for each contrast.
Parcels are coloured by their **t-statistic** (direction + magnitude of effect);
non-significant parcels are masked grey.

- **Red** → first condition > second
- **Blue** → second condition > first
- Subcortical Tian-S3 parcels shown as axial nilearn slices below the surface map.

In [ ]:
import yabplot as yab
import yabplot.data as ydata
import nibabel as nib
import tempfile, os
from nilearn import plotting

BRAIN_MAP_DIR = CONTRAST_DIR / 'brain_maps_contrast'
BRAIN_MAP_DIR.mkdir(exist_ok=True)

ALL_VIEWS = [
    'left_lateral', 'left_medial', 'right_lateral', 'right_medial',
    'superior', 'inferior', 'anterior', 'posterior',
]

try:
    _lh_surf, _rh_surf = ydata.get_surface_paths('midthickness', 'bmesh')
except Exception as e:
    print(f'yabplot surface load warning: {e}')
    _lh_surf = _rh_surf = None


def _sig_arr(result_df, value_col='t'):
    """Full-atlas array with value for FDR-sig parcels, NaN elsewhere."""
    arr = np.full(len(ALL_PARCEL_NAMES), np.nan)
    for _, row in result_df[result_df['significant']].iterrows():
        pname = row['parcel_name']
        if pname in ALL_PARCEL_NAMES:
            arr[ALL_PARCEL_NAMES.index(pname)] = row[value_col]
    return arr


def _yab_cortical(nii_path, out_png, vminmax):
    lh, rh = yab.project_vol2surf(str(nii_path), interpolation='nearest')
    lm, rm = yab.load_vertexwise_mesh(_lh_surf, _rh_surf, lh, rh)
    yab.plot_vertexwise(
        lm, rm, views=ALL_VIEWS, cmap='RdBu_r',
        vminmax=vminmax, figsize=(1600, 800),
        display_type='static', export_path=str(out_png),
        nan_color=(0.92, 0.92, 0.92),
    )
    return out_png


def _nilearn_subcortical(nii_path, out_png, vmax, title=''):
    disp = plotting.plot_stat_map(
        str(nii_path),
        display_mode='z', cut_coords=8,
        cmap='RdBu_r', vmax=vmax, symmetric_cbar=True,
        title=title, draw_cross=False,
    )
    disp.savefig(str(out_png), dpi=150)
    disp.close()
    return out_png

In [ ]:
CONTRASTS = [
    (c1, 'agreed_vs_disagreed',  'Agreed vs Disagreed'),
    (c2, 'antiright_vs_proleft', 'AntiRight vs ProLeft'),
    (c3, 'antileft_vs_proright', 'AntiLeft vs ProRight'),
]

for df, cname, label in CONTRASTS:
    n_sig = int(df['significant'].sum())
    print(f'\n{"="*55}')
    print(f'{label}  |  {n_sig} FDR-significant parcels')

    if n_sig == 0:
        print('  (nothing to plot)')
        continue

    arr  = _sig_arr(df, value_col='t')
    vext = max(float(np.nanmax(np.abs(arr))), 0.01)

    tmp = tempfile.mktemp(suffix='.nii.gz')
    parcels_to_nifti(arr, ALL_PARCEL_NAMES, ATLAS_NII, LABELS_TSV, tmp)

    # Cortical surface — yabplot
    if _lh_surf is not None:
        out_png = BRAIN_MAP_DIR / f'{cname}_cortical.png'
        _yab_cortical(tmp, out_png, vminmax=[-vext, vext])
        print(f'  Cortical   -> {out_png.name}')
    else:
        print('  (yabplot surfaces not available, skipping cortical map)')

    # Subcortical — identify Tian-S3 parcels (no '7Networks_' prefix)
    sig_names = df[df['significant']]['parcel_name'].tolist()
    subcort_sig = [p for p in sig_names if not p.startswith('7Networks_')]
    print(f'  Subcortical FDR-sig parcels ({len(subcort_sig)}): {subcort_sig}')

    out_sub = BRAIN_MAP_DIR / f'{cname}_subcortical.png'
    _nilearn_subcortical(tmp, out_sub, vmax=vext,
                         title=f'{label} — t-stat (subcortical)')
    print(f'  Subcortical -> {out_sub.name}')

    os.unlink(tmp)

## Brain Maps — Combined Contrast Figure with Significance Outlines

Produces a **single combined PNG** (`all_contrasts_combined.png`) with this layout:

**Row 1** — *Agree − Disagree*: 4 views in butterfly order  
(`LH lateral | LH medial | RH medial | RH lateral`)

**Row 2** — two 2×2 panels side by side:  
- Left panel: *Disagree — AntiLeft − ProRight*  
- Right panel: *Agree — AntiRight − ProLeft*  
Each panel: top = lateral views (LH | RH), bottom = medial views (LH | RH)

**Coloring**: Cohen’s d for all tested parcels (RdBu_r, shared symmetric scale).  
**Outlines**: FDR-significant parcels circled in yellow.  
**Colorbar**: centered at bottom, labelled *ISPC difference (Cohen’s d)*.

In [ ]:
# Imports
from yabplot.utils import load_gii
from yabplot.mesh import make_cortical_mesh, apply_dilation, get_smooth_mask
from yabplot.scene import get_view_configs, set_camera, get_shading_preset
import pyvista as pv
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.cm as cm
import matplotlib.colors as mcolors

OUTLINE_MAP_DIR = CONTRAST_DIR / 'brain_maps_outlined'
OUTLINE_MAP_DIR.mkdir(exist_ok=True)

_NAN_COLOR = (0.92, 0.92, 0.92)   # grey for untested parcels


def _make_spectral_white(n=256):
    """
    Diverging colormap: Spectral blue end -> white -> Spectral red end.
    Fixes Spectral's yellowish centre by blending each wing linearly to white.
      cmap position 0.0 = deep blue  (Spectral ~1.0)
      cmap position 0.5 = white
      cmap position 1.0 = deep red   (Spectral ~0.0)
    """
    spec  = plt.cm.get_cmap('Spectral')
    white = np.array([1.0, 1.0, 1.0, 1.0])
    half  = n // 2
    # Blue wing: deep blue (Spectral 1.0) -> white at centre
    blue = spec(np.linspace(1.0, 0.65, half)).copy()
    for i in range(half):
        t = i / (half - 1)            # 0 = deep blue, 1 = white
        blue[i] = (1 - t) * blue[i] + t * white
    # Red wing: white at centre -> deep red (Spectral 0.0)
    red = spec(np.linspace(0.35, 0.0, half)).copy()
    for i in range(half):
        t = (half - 1 - i) / (half - 1)   # 1 = white, 0 = deep red
        red[i] = (1 - t) * red[i] + t * white
    return mcolors.LinearSegmentedColormap.from_list('SpectralW', np.vstack([blue, red]))


_CMAP = _make_spectral_white()


# Array helpers

def _all_arr(result_df, value_col='cohen_d'):
    """Full-atlas array: value for every tested parcel in result_df, NaN elsewhere."""
    arr = np.full(len(ALL_PARCEL_NAMES), np.nan)
    for _, row in result_df.iterrows():
        pname = row['parcel_name']
        if pname in ALL_PARCEL_NAMES:
            arr[ALL_PARCEL_NAMES.index(pname)] = row[value_col]
    return arr


def _sig_mask_arr(result_df):
    """Full-atlas binary: 1.0 for FDR-significant parcels, NaN everywhere else."""
    arr = np.full(len(ALL_PARCEL_NAMES), np.nan)
    for _, row in result_df[result_df['significant']].iterrows():
        pname = row['parcel_name']
        if pname in ALL_PARCEL_NAMES:
            arr[ALL_PARCEL_NAMES.index(pname)] = 1.0
    return arr


# Surface geometry helpers

def _vertex_normals(vertices, faces):
    """Outward-pointing vertex normals for a triangle mesh."""
    faces_pv = np.hstack(
        [np.full((faces.shape[0], 1), 3), faces]
    ).flatten().astype(int)
    mesh = pv.PolyData(vertices, faces_pv)
    mesh.compute_normals(inplace=True, consistent_normals=True, auto_orient_normals=True)
    return mesh.point_data['Normals']


def _boundary_edges_pv(vertices, faces, sig_vals, normals, normal_offset=0.3):
    """
    PyVista PolyData of edges on the boundary between FDR-significant vertices
    (sig_vals > 0.5) and all other vertices. Edges are offset outward along
    surface normals so they render above the surface.
    """
    is_sig    = (sig_vals > 0.5).astype(np.int8)   # NaN > 0.5 -> False, correct
    e01 = np.stack([np.minimum(faces[:, 0], faces[:, 1]),
                    np.maximum(faces[:, 0], faces[:, 1])], axis=1)
    e12 = np.stack([np.minimum(faces[:, 1], faces[:, 2]),
                    np.maximum(faces[:, 1], faces[:, 2])], axis=1)
    e20 = np.stack([np.minimum(faces[:, 2], faces[:, 0]),
                    np.maximum(faces[:, 2], faces[:, 0])], axis=1)
    all_edges = np.vstack([e01, e12, e20])
    boundary  = is_sig[all_edges[:, 0]] != is_sig[all_edges[:, 1]]
    bnd_edges = np.unique(all_edges[boundary], axis=0)
    if len(bnd_edges) == 0:
        return None
    unique_ids = np.unique(bnd_edges)
    pos        = vertices[unique_ids] + normals[unique_ids] * normal_offset
    remap      = np.full(len(vertices), -1, dtype=np.intp)
    remap[unique_ids] = np.arange(len(unique_ids), dtype=np.intp)
    remapped   = remap[bnd_edges]
    n          = len(remapped)
    lines      = np.empty(n * 3, dtype=np.intp)
    lines[0::3] = 2
    lines[1::3] = remapped[:, 0]
    lines[2::3] = remapped[:, 1]
    return pv.PolyData(pos, lines=lines)


# Hemisphere mesh processing

def _process_hem(v, f, raw):
    """Mirror _render_cortical_views processing (proc_vertices=None)."""
    dilated = apply_dilation(f, raw, iterations=4)
    o_guide = get_smooth_mask(f, np.where(np.isnan(raw), 0.0, 1.0), iterations=4)
    mesh    = make_cortical_mesh(v, f, dilated)
    mesh['Slice_Mask'] = o_guide
    data_p  = mesh.clip_scalar(scalars='Slice_Mask', value=0.5, invert=False)
    base_p  = mesh.clip_scalar(scalars='Slice_Mask', value=0.5, invert=True)
    if base_p.n_points > 0:
        base_p['Data'] = np.full(base_p.n_points, np.nan)
    return base_p, [data_p]


# Single-view offscreen renderer

def _render_one_view(cfg, bases, parts, outlines, vmin, vmax_v, shading,
                     outline_color, line_width, view_px):
    """Render one brain view offscreen; return numpy array (H x W x 3)."""
    plotter = pv.Plotter(off_screen=True, window_size=(view_px, view_px), border=False)
    plotter.set_background('white')
    for b in bases:
        plotter.add_mesh(b, color=_NAN_COLOR, smooth_shading=True, **shading)
    for p in parts:
        if p.n_points == 0:
            continue
        plotter.add_mesh(
            p, scalars='Data', cmap=_CMAP, clim=(vmin, vmax_v),
            n_colors=256, nan_color=_NAN_COLOR, show_scalar_bar=False,
            smooth_shading=True, **shading,
        )
    for ol in outlines:
        plotter.add_mesh(ol, color=outline_color, line_width=line_width,
                         render_lines_as_tubes=False)
    set_camera(plotter, cfg, zoom=1.2)
    plotter.hide_axes()
    img = plotter.screenshot(return_img=True)
    plotter.close()
    return img



# White-margin cropper (removes blank border from each rendered view)

def _crop_margins(img, pad=6):
    """Crop white border from an H x W x 3 numpy array; keep a small pad."""
    mask = np.any(img < 248, axis=2)   # True where pixel is non-white
    if not mask.any():
        return img
    rows = np.where(mask.any(axis=1))[0]
    cols = np.where(mask.any(axis=0))[0]
    r0 = max(0, rows[0]  - pad)
    r1 = min(img.shape[0], rows[-1] + pad + 1)
    c0 = max(0, cols[0]  - pad)
    c1 = min(img.shape[1], cols[-1] + pad + 1)
    return img[r0:r1, c0:c1]


# Per-contrast render -> dict of numpy arrays

def render_contrast_views(nii_all_path, nii_sig_path, vminmax,
                           outline_color='yellow', line_width=2.0,
                           normal_offset=0.3, view_px=500):
    """
    Project and render one contrast into 4 brain views.
    Returns dict: view_name -> numpy array (H x W x 3).
    Keys: 'left_lateral', 'left_medial', 'right_lateral', 'right_medial'.
    """
    vmin, vmax_v = vminmax
    lh_t,   rh_t   = yab.project_vol2surf(str(nii_all_path), interpolation='nearest')
    lh_sig, rh_sig = yab.project_vol2surf(str(nii_sig_path), interpolation='nearest')
    lh_v, lh_f = load_gii(_lh_surf)
    rh_v, rh_f = load_gii(_rh_surf)
    lh_normals  = _vertex_normals(lh_v, lh_f)
    rh_normals  = _vertex_normals(rh_v, rh_f)
    lh_outline  = _boundary_edges_pv(lh_v, lh_f, lh_sig, lh_normals, normal_offset)
    rh_outline  = _boundary_edges_pv(rh_v, rh_f, rh_sig, rh_normals, normal_offset)
    lh_base, lh_parts = _process_hem(lh_v, lh_f, lh_t)
    rh_base, rh_parts = _process_hem(rh_v, rh_f, rh_t)
    view_cfgs   = get_view_configs(['left_lateral', 'left_medial',
                                    'right_lateral', 'right_medial'])
    shading     = get_shading_preset('default')
    views       = {}
    for vname, cfg in view_cfgs.items():
        is_left  = cfg['side'] == 'L'
        bases    = ([lh_base] if is_left and lh_base.n_points > 0 else
                    [rh_base] if not is_left and rh_base.n_points > 0 else [])
        parts    = lh_parts if is_left else rh_parts
        outlines = ([lh_outline] if is_left and lh_outline is not None else
                    [rh_outline] if not is_left and rh_outline is not None else [])
        views[vname] = _crop_margins(_render_one_view(
            cfg, bases, parts, outlines, vmin, vmax_v, shading,
            outline_color, line_width, view_px,
        ))
    return views


# Figure assembly

def build_combined_figure(views_c1, views_c2, views_c3, global_vext,
                           out_path=None, figsize=(14, 9), dpi=150,
                           cbar_label="ISPC difference  (Cohen's d)"):
    """
    Three contrasts, no headings, maximum brain density.

    Row 0  Contrast 1 (Agree-Disagree): 4 butterfly views
    Row 1  Top of 2x2 panels: lateral views (c3 left | c3 right | c2 left | c2 right)
    Row 2  Bottom of 2x2 panels: medial views (same column order)
    Row 3  Shared colorbar (centered)
    """
    ROW1_ORDER = ['left_lateral', 'left_medial', 'right_medial', 'right_lateral']
    GRID_ROWS  = [('left_lateral', 'right_lateral'),
                  ('left_medial',  'right_medial')]

    fig = plt.figure(figsize=figsize, dpi=dpi, facecolor='white')

    gs = gridspec.GridSpec(
        4, 4,
        figure=fig,
        height_ratios=[1.0, 1.0, 1.0, 0.07],
        hspace=0.0,
        wspace=0.0,
        left=0.002, right=0.998, top=0.998, bottom=0.07,
    )

    # Row 0: contrast 1 butterfly
    for col_i, vname in enumerate(ROW1_ORDER):
        ax = fig.add_subplot(gs[0, col_i])
        ax.imshow(views_c1[vname], aspect='auto')
        ax.axis('off')

    # Rows 1-2: contrast 3 (cols 0-1) and contrast 2 (cols 2-3)
    panel_views = [views_c3, views_c2]
    for row_i, (vname_l, vname_r) in enumerate(GRID_ROWS):
        for panel_i, pviews in enumerate(panel_views):
            col_l = panel_i * 2
            col_r = panel_i * 2 + 1
            ax_l = fig.add_subplot(gs[1 + row_i, col_l])
            ax_l.imshow(pviews[vname_l], aspect='auto')
            ax_l.axis('off')
            ax_r = fig.add_subplot(gs[1 + row_i, col_r])
            ax_r.imshow(pviews[vname_r], aspect='auto')
            ax_r.axis('off')

    # Colorbar — placed via add_axes for precise control independent of the brain grid
    norm   = mcolors.Normalize(vmin=-global_vext, vmax=global_vext)
    sm     = cm.ScalarMappable(cmap=_CMAP, norm=norm)
    sm.set_array([])
    cbar_ax = fig.add_axes([0.25, 0.015, 0.50, 0.025])
    cb = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal')
    cb.outline.set_visible(False)                          # remove black frame
    cb.ax.tick_params(labelsize=14, length=2.5,
                      color='#555555', labelcolor='#333333')
    cb.set_label(cbar_label,
                 fontsize=14, color='#333333', labelpad=5,
                 fontfamily='DM Sans')
    cb.set_ticks(np.linspace(-global_vext, global_vext, 5))
    cb.ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.2f}'))

    if out_path:
        fig.savefig(str(out_path), dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.show()
    return fig

# Render loop

if _lh_surf is None:
    print('yabplot surfaces not available - skipping.')
else:
    # Shared symmetric vmax across all three contrasts
    global_vext = max(
        float(np.nanmax(np.abs(_all_arr(c1)))),
        float(np.nanmax(np.abs(_all_arr(c2)))),
        float(np.nanmax(np.abs(_all_arr(c3)))),
        0.01,
    )
    vminmax = (-global_vext, global_vext)
    print(f"Global symmetric vmax (Cohen's d): {global_vext:.3f}\n")

    rendered = {}
    for df, cname, label in CONTRASTS:
        print(f'Rendering {label} ...')
        arr_all = _all_arr(df)
        arr_sig = _sig_mask_arr(df)
        tmp_all = tempfile.mktemp(suffix='.nii.gz')
        tmp_sig = tempfile.mktemp(suffix='.nii.gz')
        parcels_to_nifti(arr_all, ALL_PARCEL_NAMES, ATLAS_NII, LABELS_TSV, tmp_all)
        parcels_to_nifti(arr_sig, ALL_PARCEL_NAMES, ATLAS_NII, LABELS_TSV, tmp_sig)
        rendered[cname] = render_contrast_views(tmp_all, tmp_sig, vminmax)
        os.unlink(tmp_all)
        os.unlink(tmp_sig)
        print(f'  done.\n')

    out_png = OUTLINE_MAP_DIR / 'all_contrasts_combined.png'
    build_combined_figure(
        views_c1    = rendered['agreed_vs_disagreed'],
        views_c2    = rendered['antiright_vs_proleft'],
        views_c3    = rendered['antileft_vs_proright'],
        global_vext = global_vext,
        out_path    = out_png,
    )
    print(f'Saved -> {out_png}')

## Save Results

In [ ]:
for cname, df in [
    ('agreed_vs_disagreed',  c1),
    ('antiright_vs_proleft', c2),
    ('antileft_vs_proright', c3),
]:
    out = CONTRAST_DIR / f'contrast_{cname}_subject_level.csv'
    df.to_csv(out, index=False)
    print(f'{cname}: {len(df)} parcels, {df["significant"].sum()} FDR-sig -> {out.name}')

---
## Section B — 2×2 Repeated-Measures ANOVA (Subject Level)

Tests three orthogonal effects in a single model using the full 2×2 factorial structure:

| | **Pro** | **Anti** |
|---|---|---|
| **Agreed** | ProLeft | AntiRight |
| **Disagreed** | ProRight | AntiLeft |

- **Agreement** (Agreed vs Disagreed): ISPC for content matching vs contradicting subject's views
- **Group** (Ingroup/Left vs Outgroup/Right): ISPC for content about the ingroup vs outgroup
- **Interaction** (Agreement × Group = Pro vs Anti): Valence emerges here — whether ISPC for Pro vs Anti content differs by group alignment

**Method:** Vectorized paired t-tests (F = t² for df1=1 balanced designs — mathematically equivalent
to a full RM-ANOVA). FDR correction (BH, q=0.05) applied independently per effect.
**Effect size:** signed Cohen's dz (within-subjects). Parcels selected by same criterion as Contrast 1.

In [ ]:
anova_parcels = get_parcels_for_anova()
print(f'Parcels entering ANOVA: {len(anova_parcels)}')

anova_df = run_anova_parcelwise(subj_arr, anova_parcels)
display(anova_df.head(15))

In [ ]:
# Box plots for Agreement and Group effects (default style)
for eff, label in [
    ('agreement', 'Agreement Main Effect (Agreed vs Disagreed)'),
    ('group',     'Group Main Effect (Ingroup vs Outgroup)'),
]:
    box_plot_anova(
        anova_df, eff, subj_arr, ALL_PARCEL_NAMES,
        title=f'2x2 RM-ANOVA — {label}',
        out_path=ANOVA_DIR / f'boxplot_{eff}.png',
    )

# Interaction (Pro vs Anti): top 5 parcels, custom colours, APA-style effect markers
box_plot_anova(
    anova_df, 'interaction', subj_arr, ALL_PARCEL_NAMES,
    n_parcels=5,
    color_agreed='#77a650',
    color_disagreed='#bb4e31',
    mark_other_effects=True,
    title='2x2 RM-ANOVA — Agreement × Group Interaction (Pro vs Anti)',
    out_path=ANOVA_DIR / 'boxplot_interaction.png',
)

In [ ]:
if _lh_surf is None:
    print('yabplot surfaces not available - skipping.')
else:
    ANOVA_MAP_DIR = ANOVA_DIR / 'brain_maps'
    ANOVA_MAP_DIR.mkdir(exist_ok=True)

    ANOVA_EFFECTS = [
        ('agreement',   'Agreement main effect'),
        ('group',     'Group main effect (Ingroup vs Outgroup)'),
        ('interaction', 'Agreement x Valence interaction'),
    ]

    # Shared symmetric vmax across all three effects
    global_vext_anova = max(
        float(np.nanmax(np.abs(_all_arr(make_anova_view_df(anova_df, 'agreement'))))),
        float(np.nanmax(np.abs(_all_arr(make_anova_view_df(anova_df, 'group'))))),
        float(np.nanmax(np.abs(_all_arr(make_anova_view_df(anova_df, 'interaction'))))),
        0.01,
    )
    vminmax_anova = (-global_vext_anova, global_vext_anova)
    print(f"Global symmetric vmax (Cohen's dz): {global_vext_anova:.3f}\n")

    rendered_anova = {}
    for eff, label in ANOVA_EFFECTS:
        print(f'Rendering {label} ...')
        view_df  = make_anova_view_df(anova_df, eff)
        arr_all  = _all_arr(view_df)
        arr_sig  = _sig_mask_arr(view_df)
        tmp_all  = tempfile.mktemp(suffix='.nii.gz')
        tmp_sig  = tempfile.mktemp(suffix='.nii.gz')
        parcels_to_nifti(arr_all, ALL_PARCEL_NAMES, ATLAS_NII, LABELS_TSV, tmp_all)
        parcels_to_nifti(arr_sig, ALL_PARCEL_NAMES, ATLAS_NII, LABELS_TSV, tmp_sig)
        rendered_anova[eff] = render_contrast_views(tmp_all, tmp_sig, vminmax_anova)
        os.unlink(tmp_all)
        os.unlink(tmp_sig)
        print(f'  done.\n')

    # Simple effects: Pro vs Anti within each Agreement group
    print('Rendering simple effects...')
    se_agreed_df    = run_contrast(subj_arr['ProLeft'],  subj_arr['AntiRight'],
                                   'ProLeft', 'AntiRight', anova_parcels)
    se_disagreed_df = run_contrast(subj_arr['ProRight'], subj_arr['AntiLeft'],
                                   'ProRight', 'AntiLeft', anova_parcels)
    print(f"  SE Agreed    (ProLeft>AntiRight):   {se_agreed_df['significant'].sum()} FDR-sig")
    print(f"  SE Disagreed (ProRight>AntiLeft):   {se_disagreed_df['significant'].sum()} FDR-sig")

    for df, key in [
        (se_agreed_df,    'se_agreed'),
        (se_disagreed_df, 'se_disagreed'),
    ]:
        arr_all = _all_arr(df)
        arr_sig = _sig_mask_arr(df)
        tmp_all = tempfile.mktemp(suffix='.nii.gz')
        tmp_sig = tempfile.mktemp(suffix='.nii.gz')
        parcels_to_nifti(arr_all, ALL_PARCEL_NAMES, ATLAS_NII, LABELS_TSV, tmp_all)
        parcels_to_nifti(arr_sig, ALL_PARCEL_NAMES, ATLAS_NII, LABELS_TSV, tmp_sig)
        rendered_anova[key] = render_contrast_views(tmp_all, tmp_sig, vminmax_anova)
        os.unlink(tmp_all)
        os.unlink(tmp_sig)
        print(f'  {key}: done.\n')

In [ ]:
if _lh_surf is not None:
    out_png_anova = ANOVA_DIR / 'brain_maps' / 'anova_effects_combined.png'
    build_combined_figure(
        views_c1    = rendered_anova['agreement'],
        views_c2    = rendered_anova['se_disagreed'],  # bottom right: ProRight vs AntiLeft
        views_c3    = rendered_anova['se_agreed'],     # bottom left:  ProLeft  vs AntiRight
        global_vext = global_vext_anova,
        out_path    = out_png_anova,
        cbar_label  = "ISPC difference  (Cohen's dz)",
    )
    print(f'Saved -> {out_png_anova}', f'cmap {_CMAP}')

## Save ANOVA Results

In [ ]:
anova_df.to_csv(ANOVA_DIR / 'parcelwise_anova_2x2.csv', index=False)
print(f'Saved ANOVA results: {len(anova_df)} parcels')
print(f'  Agreement:   {anova_df["sig_agreement"].sum()} FDR-sig')
print(f'  Group:       {anova_df["sig_group"].sum()} FDR-sig')
print(f'  Interaction: {anova_df["sig_interaction"].sum()} FDR-sig')

---
## Section C — Pro vs Anti Valence Contrast

Collapses across the Agreement dimension to test the pure **valence** effect:
do left-wing subjects synchronise more while watching **pro-left content**
(ProLeft + ProRight averaged) than **anti-left content** (AntiLeft + AntiRight averaged)?

- **Pro** = (ProLeft + ProRight) / 2 per subject per parcel
- **Anti** = (AntiLeft + AntiRight) / 2 per subject per parcel
- Positive `mean_diff` = higher ISPC for Pro content.

Parcel criterion: FDR-significant AND |ISC| ≥ 0.1 in any of the 4 conditions (same as Contrast 1 and ANOVA).

In [ ]:
parcels_c4 = get_parcels_for_contrast(['AntiLeft', 'AntiRight', 'ProLeft', 'ProRight'])
print(f'Parcels selected for Pro vs Anti: {len(parcels_c4)}')

pro_subj  = (subj_arr['ProLeft']  + subj_arr['ProRight'])  / 2
anti_subj = (subj_arr['AntiLeft'] + subj_arr['AntiRight']) / 2

c4 = run_contrast(pro_subj, anti_subj, 'Pro', 'Anti', parcels_c4)
print(f'Pro vs Anti  |  {len(c4)} parcels tested  |  {c4["significant"].sum()} FDR-sig')
display(c4.head(15))

In [ ]:
box_plot(
    c4, pro_subj, anti_subj, 'Pro', 'Anti',
    color_a='#77a650', color_b='#bb4e31',
    title='Section C: Pro vs Anti — Subject-Level LOO-ISPC',
    out_path=CONTRAST_DIR / 'subj_contrast_pro_vs_anti.png',
)

### Brain Maps — Pro vs Anti

Parcels coloured by t-statistic (red = Pro > Anti, blue = Anti > Pro).

In [ ]:
PRO_ANTI_MAP_DIR = CONTRAST_DIR / 'brain_maps_pro_vs_anti'
PRO_ANTI_MAP_DIR.mkdir(exist_ok=True)

n_sig = int(c4['significant'].sum())
print(f'Pro vs Anti  |  {n_sig} FDR-significant parcels')

if n_sig > 0:
    arr  = _sig_arr(c4, value_col='t')
    vext = max(float(np.nanmax(np.abs(arr[~np.isnan(arr)]))), 0.01)

    tmp = tempfile.mktemp(suffix='.nii.gz')
    parcels_to_nifti(arr, ALL_PARCEL_NAMES, ATLAS_NII, LABELS_TSV, tmp)

    if _lh_surf is not None:
        out_png = PRO_ANTI_MAP_DIR / 'pro_vs_anti_cortical.png'
        _yab_cortical(tmp, out_png, vminmax=[-vext, vext])
        print(f'  Cortical   -> {out_png.name}')
    else:
        print('  (yabplot surfaces not available, skipping cortical map)')

    subcort_sig = [p for p in c4[c4['significant']]['parcel_name'] if not p.startswith('7Networks_')]
    print(f'  Subcortical FDR-sig parcels ({len(subcort_sig)}): {subcort_sig}')

    out_sub = PRO_ANTI_MAP_DIR / 'pro_vs_anti_subcortical.png'
    _nilearn_subcortical(tmp, out_sub, vmax=vext, title='Pro vs Anti — t-stat (subcortical)')
    print(f'  Subcortical -> {out_sub.name}')

    os.unlink(tmp)
else:
    print('  No FDR-significant parcels — skipping brain maps')

### Save Results

In [ ]:
out = CONTRAST_DIR / 'contrast_pro_vs_anti_subject_level.csv'
c4.to_csv(out, index=False)
print(f'Pro vs Anti: {len(c4)} parcels, {c4["significant"].sum()} FDR-sig -> {out.name}')

---
## Section D — Neurosynth Decoding Prep

For each of the 3 t-test contrasts, this cell:
1. Loads the saved per-contrast results CSVs
2. Computes MNI centroids for every significant parcel
3. Assigns anatomical labels from the local AAL2 atlas (largest-overlap region)
4. Saves a binary NIfTI mask: one **combined** mask per contrast (upload to Neurosynth ROI decoder) + one mask per individual parcel
5. Saves a **CSV** table with parcel name, network, hemisphere, subregion, AAL label, MNI x/y/z, Cohen's d, direction, and FDR-corrected p

Output layout:
```
contrasts/neurosynth/
  proleft_vs_antiright/
    mask_combined.nii.gz       ← upload this to neurosynth.org/decode/
    parcels/
      7Networks_LH_Vis_4.nii.gz
      ...
    parcel_locations.csv
  proright_vs_antileft/  ...
  agreed_vs_disagreed/   ...
  all_contrasts_parcel_locations.csv
```

In [ ]:
import nibabel as nib
import numpy as np
import pandas as pd
from scipy import ndimage
from nilearn.image import resample_to_img

# ── Atlas ──────────────────────────────────────────────────────────────────────
_atlas_img    = nib.load(ATLAS_NII)
_atlas_data   = np.asarray(_atlas_img.dataobj)
_atlas_affine = _atlas_img.affine

_labels_df  = pd.read_csv(LABELS_TSV, sep='\t')
_name_to_id = dict(zip(_labels_df['name'], _labels_df['id']))

# ── AAL2 atlas (local copy) ────────────────────────────────────────────────────
AAL2_NII = ROOT / 'data/atlases/aal2_for_SPM12/aal/aal2.nii.gz'
AAL2_TXT = ROOT / 'data/atlases/aal2_for_SPM12/aal/aal2.nii.txt'

_aal_rs   = resample_to_img(nib.load(AAL2_NII), _atlas_img,
                             interpolation='nearest',
                             force_resample=True, copy_header=True)
_aal_data = np.asarray(_aal_rs.dataobj).astype(int)

_aal_labels = {}  # voxel integer value → region name
with open(AAL2_TXT) as _fh:
    for _line in _fh:
        _p = _line.strip().split()
        _aal_labels[int(_p[0])] = _p[1]

# ── Output directory ───────────────────────────────────────────────────────────
NEUROSYNTH_DIR = CONTRAST_DIR / 'neurosynth'
NEUROSYNTH_DIR.mkdir(exist_ok=True)

# ── Contrast definitions ───────────────────────────────────────────────────────
# cond_A / cond_B: match the saved CSV column ordering (mean_diff = A - B)
# negative cohen_d → cond_B > cond_A
_CONTRASTS = [
    dict(
        name   = 'proleft_vs_antiright',
        label  = 'ProLeft > AntiRight (within Agreed)',
        csv    = CONTRAST_DIR / 'contrast_antiright_vs_proleft_subject_level.csv',
        cond_A = 'AntiRight',
        cond_B = 'ProLeft',
    ),
    dict(
        name   = 'proright_vs_antileft',
        label  = 'ProRight > AntiLeft (within Disagreed)',
        csv    = CONTRAST_DIR / 'contrast_antileft_vs_proright_subject_level.csv',
        cond_A = 'AntiLeft',
        cond_B = 'ProRight',
    ),
    dict(
        name   = 'agreed_vs_disagreed',
        label  = 'Agreed vs Disagreed',
        csv    = CONTRAST_DIR / 'contrast_agreed_vs_disagreed_subject_level.csv',
        cond_A = 'Agreed',
        cond_B = 'Disagreed',
    ),
]

# ── Helper functions ───────────────────────────────────────────────────────────
def _parse_parcel(name):
    """Return (hemisphere, network, subregion) from a 7Networks parcel name."""
    parts = name.split('_')
    hemi  = parts[1]
    net   = parts[2]
    sub_parts = parts[3:]
    # sub_parts[-1] is the numeric suffix; everything before is the subregion
    subregion = '_'.join(sub_parts[:-1]) if len(sub_parts) > 1 else ''
    return hemi, net, subregion or net

def _mni_centroid(pid):
    mask = _atlas_data == pid
    if not mask.any():
        return np.full(3, np.nan)
    vox = np.array(ndimage.center_of_mass(mask))
    return np.round(nib.affines.apply_affine(_atlas_affine, vox), 1)

def _top_aal(pid):
    mask = _atlas_data == pid
    vals = _aal_data[mask]
    vals = vals[vals > 0]
    if len(vals) == 0:
        return 'no_overlap'
    uniq, counts = np.unique(vals, return_counts=True)
    return _aal_labels.get(int(uniq[np.argmax(counts)]), 'unknown')

def _save_mask(parcel_ids, path):
    vol = np.zeros(_atlas_data.shape, dtype=np.uint8)
    for pid in parcel_ids:
        vol[_atlas_data == pid] = 1
    nib.save(nib.Nifti1Image(vol, _atlas_affine, _atlas_img.header), path)

def _save_effect_map(parcel_id_to_d, path):
    """Cohen's d map — each parcel's voxels filled with its cohen_d value.
    This is what you upload to NeuroVault for Neurosynth decoding."""
    vol = np.zeros(_atlas_data.shape, dtype=np.float32)
    for pid, d in parcel_id_to_d.items():
        vol[_atlas_data == pid] = d
    nib.save(nib.Nifti1Image(vol, _atlas_affine, _atlas_img.header), path)

# ── Main loop ──────────────────────────────────────────────────────────────────
all_rows = []

for spec in _CONTRASTS:
    df  = pd.read_csv(spec['csv'])
    sig = df[df['significant'] == True].copy()
    print(f"\n{'='*62}")
    print(f"{spec['label']}  —  {len(sig)} significant parcels")
    print('='*62)

    out_dir     = NEUROSYNTH_DIR / spec['name']
    parcels_dir = out_dir / 'parcels'
    out_dir.mkdir(exist_ok=True)
    parcels_dir.mkdir(exist_ok=True)

    rows, pids = [], []

    for _, row in sig.iterrows():
        pname = row['parcel_name']
        pid   = _name_to_id.get(pname)
        if pid is None:
            print(f'  WARNING: {pname} not in atlas labels — skipped')
            continue

        pids.append(pid)
        hemi, net, sub = _parse_parcel(pname)
        mni = _mni_centroid(pid)
        d   = row['cohen_d']

        rows.append({
            'contrast':    spec['label'],
            'parcel_name': pname,
            'hemisphere':  hemi,
            'network':     net,
            'subregion':   sub,
            'aal_region':  _top_aal(pid),
            'mni_x': mni[0], 'mni_y': mni[1], 'mni_z': mni[2],
            'cohen_d':   round(d, 4),
            'direction': f"{spec['cond_B']} > {spec['cond_A']}" if d < 0
                         else f"{spec['cond_A']} > {spec['cond_B']}",
            'p_fdr':     row['p_fdr'],
        })

        _save_mask([pid], parcels_dir / f'{pname}.nii.gz')
    
    pid_to_d = {_name_to_id[r['parcel_name']]: r['cohen_d'] for r in rows}
    _save_mask(pids, out_dir / 'mask_combined.nii.gz')
    _save_effect_map(pid_to_d, out_dir / 'cohend_map.nii.gz')


    cdf = pd.DataFrame(rows)
    cdf.to_csv(out_dir / 'parcel_locations.csv', index=False)

    print(f"  Combined mask → {out_dir.relative_to(ROOT)}/mask_combined.nii.gz")
    print(f"  Parcel masks  → {len(pids)} files in parcels/")
    print(f"  Effect map    → {out_dir.relative_to(ROOT)}/cohend_map.nii.gz")
    print(f"  CSV           → {out_dir.relative_to(ROOT)}/parcel_locations.csv")
    print()
    display(
        cdf[['parcel_name','network','hemisphere','subregion',
             'aal_region','mni_x','mni_y','mni_z','cohen_d','direction','p_fdr']]
        .sort_values('cohen_d')
        .reset_index(drop=True)
    )

    all_rows.extend(rows)

# ── Master CSV ─────────────────────────────────────────────────────────────────
master_csv = NEUROSYNTH_DIR / 'all_contrasts_parcel_locations.csv'
pd.DataFrame(all_rows).to_csv(master_csv, index=False)
print(f'Master CSV → {master_csv.relative_to(ROOT)}')


In [ ]:
import json

DECODER_DIR = ROOT / 'data/neurosynth'
TOP_N       = 25

_DECODE_CONTRASTS = [
    ('proleft_vs_antiright',  'ProLeft > AntiRight (within Agreed)'),
    ('proright_vs_antileft',  'ProRight > AntiLeft (within Disagreed)'),
    ('agreed_vs_disagreed',   'Agreed vs Disagreed'),
]

for cname, clabel in _DECODE_CONTRASTS:
    json_path = DECODER_DIR / f'{cname}.json'
    if not json_path.exists():
        print(f'[MISSING]  {json_path}')
        continue

    with open(json_path) as _f:
        raw = json.load(_f)['data']['values']
    raw.pop('', None)  # drop empty-string key if present

    s = pd.Series(raw).sort_values(ascending=False)

    top = s.head(TOP_N)
    bot = s.tail(TOP_N)

    print(f'{"="*62}')
    print(f'  {clabel}')
    print(f'{"="*62}')

    tbl = pd.concat([
        top.rename('r').reset_index().rename(columns={'index': 'term'}).assign(direction='positive'),
        bot.sort_values().rename('r').reset_index().rename(columns={'index': 'term'}).assign(direction='negative'),
    ], ignore_index=True)
    display(tbl)

    fig, (ax_pos, ax_neg) = plt.subplots(1, 2, figsize=(14, 5))

    top_sorted = top.sort_values()
    ax_pos.barh(top_sorted.index, top_sorted.values, color='steelblue')
    ax_pos.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax_pos.set_xlabel('Correlation')
    ax_pos.set_title(f'Top {TOP_N} positive')

    bot_sorted = bot.sort_values(ascending=False)
    ax_neg.barh(bot_sorted.index, bot_sorted.values, color='tomato')
    ax_neg.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax_neg.set_xlabel('Correlation')
    ax_neg.set_title(f'Top {TOP_N} negative')

    fig.suptitle(clabel, fontsize=12, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()


In [ ]:
# ── ISPC Presentation Illustration ──────────────────────────────────────────
# Parcel: 7Networks_RH_Default_Temp_4 | Condition: ProLeft | Subject: sub-1
# 4 panels: subject matrix / subject vector / others-avg matrix / others-avg vector
import glob

PARC_DIR = ROOT / 'data/derivatives/parcellated'
PARCEL   = '7Networks_RH_Default_Temp_4'
COND     = 'ProLeft'
CMAP     = 'Spectral'

# Load time series for all subjects
_files = sorted(glob.glob(str(PARC_DIR / f'sub-*/*{COND}*.tsv')))
_all_ts = np.array([
    pd.read_csv(f, sep='\t')[PARCEL].values for f in _files
])                                  # (n_subjects, n_timepoints=267)

T = _all_ts.shape[1]                # 267

# Subject 0 (sub-1) vs. average of all others
_subj_raw   = _all_ts[0]
_others_raw = _all_ts[1:].mean(axis=0)

# Z-score independently
def _zs(x): return (x - x.mean()) / (x.std() + 1e-12)
subj_z   = _zs(_subj_raw)
others_z = _zs(_others_raw)

# Shared symmetric color limits
_vmax = max(np.abs(subj_z).max(), np.abs(others_z).max())

# Matrix: pad T → 270 (15 × 18) so time flows left-to-right, row by row
_nr, _nc = 15, 18
_pad = _nr * _nc - T
def _to_mat(z): return np.pad(z, (0, _pad)).reshape(_nr, _nc)
subj_mat   = _to_mat(subj_z)
others_mat = _to_mat(others_z)

# Vector: 1 × T horizontal strip (time left-to-right)
subj_vec   = subj_z.reshape(1, T)
others_vec = others_z.reshape(1, T)

# ── Figure: 2 rows × 2 cols, no labels, no colorbar ─────────────────────────
fig, axes = plt.subplots(
    2, 2,
    figsize=(9, 6),
    gridspec_kw=dict(
        height_ratios=[_nr, 2],     # matrix row much taller than vector strip
        hspace=0.04, wspace=0.04,
        left=0.01, right=0.99, top=0.99, bottom=0.01,
    ),
    facecolor='white',
)

_kw = dict(cmap=CMAP, vmin=-_vmax, vmax=_vmax, aspect='auto', interpolation='nearest')

axes[0, 0].imshow(subj_mat,   **_kw)   # subject — matrix
axes[0, 1].imshow(others_mat, **_kw)   # others avg — matrix
axes[1, 0].imshow(subj_vec,   **_kw)   # subject — vector
axes[1, 1].imshow(others_vec, **_kw)   # others avg — vector

for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

_out = OUTPUT_DIR / 'contrasts' / 'anova' / 'ispc_illustration.png'
fig.savefig(str(_out), dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved -> {_out}')

In [ ]:
# ── 4-condition ANOVA box plot — LH_SalVentAttn_FrOper_3 ─────────────────────
# Conditions: Pro Ingroup (ProLeft), Anti Outgroup (AntiRight),
#             Pro Outgroup (ProRight), Anti Ingroup (AntiLeft)
# Notebook colour theme; all subjects shown with gray connectors
# Significance brackets: Group (nested) + Interaction (Pro vs Anti arc)

_PARCEL = '7Networks_LH_SalVentAttn_FrOper_3'
_idx    = ALL_PARCEL_NAMES.index(_PARCEL)
_row    = anova_df[anova_df['parcel_name'] == _PARCEL].iloc[0]
_n      = len(common_subjects)

_COND_ORDER  = ['ProLeft',      'AntiRight',     'ProRight',     'AntiLeft']
_COND_LABELS = ['Pro\nIngroup', 'Anti\nOutgroup', 'Pro\nOutgroup', 'Anti\nIngroup']
_COLORS      = ["#8fbb6b",      "#5c8a36",       "#c58877",      '#bb4e31']

_data = [subj_arr[c][:, _idx] for c in _COND_ORDER]
_rng  = np.random.default_rng(42)

fig, ax = plt.subplots(figsize=(7, 7))

# Box plot
_bp = ax.boxplot(
    _data, positions=[0,1,2,3], widths=0.45, patch_artist=True,
    medianprops  = dict(color='black', linewidth=2.0),
    boxprops     = dict(linewidth=1.2),
    whiskerprops = dict(linewidth=1.0),
    capprops     = dict(linewidth=1.0),
    flierprops   = dict(marker=''),
    zorder=2,
)
for patch, col in zip(_bp['boxes'], _COLORS):
    patch.set_facecolor(col); patch.set_alpha(1)

# Gray lines (all 4 connected per subject)
for s in range(_n):
    ax.plot([0,1,2,3], [d[s] for d in _data],
            color='grey', alpha=0.18, linewidth=0.7, zorder=1)

# Scatter dots
for xi, d in enumerate(_data):
    jitter = _rng.uniform(-0.08, 0.08, _n)
    ax.scatter(xi + jitter, d, color='black', s=18, alpha=0.55, zorder=3)

ax.set_xticks([0,1,2,3])
ax.set_xticklabels(_COND_LABELS, fontsize=10)
ax.set_ylabel('LOO-ISPC', fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

_y_all = np.concatenate(_data)
_ymax, _ymin = _y_all.max(), _y_all.min()
_yr   = _ymax - _ymin
_col  = '#333333'
_lw   = 1.1

def _sig_stars(p):
    return '***' if p < 0.001 else '**' if p < 0.01 else '*'

def _hbracket(ax, x0, x1, y, tk):
    ax.plot([x0, x1], [y, y],   color=_col, lw=_lw, clip_on=False)
    ax.plot([x0, x0], [y-tk, y], color=_col, lw=_lw, clip_on=False)
    ax.plot([x1, x1], [y-tk, y], color=_col, lw=_lw, clip_on=False)

_tk = _yr * 0.025

# ── Group bracket (nested: Outgroup inner, Ingroup outer) ────────────────────
if _row['sig_group']:
    _y_og = _ymax + _yr * 0.13
    _y_ig = _ymax + _yr * 0.25
    _hbracket(ax, 0.55, 2.45, _y_og, _tk)    # Outgroup: boxes 1–2
    _hbracket(ax, -0.45, 3.45, _y_ig, _tk)   # Ingroup:  boxes 0–3
    ax.text(1.5, (_y_og + _y_ig)/2, _sig_stars(_row['p_fdr_group']),
            ha='center', va='center', fontsize=11, color=_col, fontweight='bold')
    ax.text(1.5, _y_ig + _tk * 0.8, f'p_fdr = {_row["p_fdr_group"]:.3f}',
            ha='center', va='bottom', fontsize=7.5, color=_col)

# ── Interaction bracket (Pro arc vs Anti arc) ────────────────────────────────
if _row['sig_interaction']:
    _y0 = _ymax + _yr * (0.38 if _row['sig_group'] else 0.13)
    _y1 = _y0 + _yr * 0.12

    # Pro arc: connects boxes 0 and 2 at y0
    ax.plot([-0.45, 2.45], [_y0, _y0], color=_col, lw=_lw, clip_on=False)
    ax.plot([-0.45, -0.45], [_y0-_tk, _y0], color=_col, lw=_lw, clip_on=False)
    ax.plot([2.45, 2.45],   [_y0-_tk, _y0], color=_col, lw=_lw, clip_on=False)

    # Anti arc: connects boxes 1 and 3 at y1
    ax.plot([0.55, 3.45], [_y1, _y1], color=_col, lw=_lw, clip_on=False)
    ax.plot([0.55, 0.55], [_y1-_tk, _y1], color=_col, lw=_lw, clip_on=False)
    ax.plot([3.45, 3.45], [_y1-_tk, _y1], color=_col, lw=_lw, clip_on=False)

    ax.text(1.5, (_y0 + _y1)/2, _sig_stars(_row['p_fdr_interaction']),
            ha='center', va='center', fontsize=11, color=_col, fontweight='bold')
    ax.text(1.5, _y1 + _tk * 0.8, f'p_fdr = {_row["p_fdr_interaction"]:.3f}',
            ha='center', va='bottom', fontsize=7.5, color=_col)

_y_top = _ymax + _yr * (0.66 if (_row['sig_group'] and _row['sig_interaction']) else 0.42)
ax.set_ylim(_ymin - _yr * 0.06, _y_top)

plt.tight_layout()
_out = ANOVA_DIR / 'anova_boxplot_4cond.png'
fig.savefig(str(_out), dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved -> {_out}')

In [ ]:
# ── 4-condition ANOVA box plot — RH_Vis_21 (largest interpretable condition spread) ──
# sig_interaction; range≈0.128; all 4 conditions clearly positive (0.17–0.30); Pro>Anti

_PARCEL2 = '7Networks_RH_Vis_21'
_idx2    = ALL_PARCEL_NAMES.index(_PARCEL2)
_row2    = anova_df[anova_df['parcel_name'] == _PARCEL2].iloc[0]
_n2      = len(common_subjects)

_COND_ORDER2  = ['ProLeft',      'AntiRight',     'ProRight',     'AntiLeft']
_COND_LABELS2 = ['Pro\nIngroup', 'Anti\nOutgroup', 'Pro\nOutgroup', 'Anti\nIngroup']
_COLORS2      = ['#aec6e8',      '#5a8fc2',       '#f2b3a0',      '#c0504d']

_data2 = [subj_arr[c][:, _idx2] for c in _COND_ORDER2]
_rng2  = np.random.default_rng(42)

fig2, ax2 = plt.subplots(figsize=(11, 5))
fig2.patch.set_facecolor('white')

ax2.set_axisbelow(True)

_bp2 = ax2.boxplot(
    _data2, positions=[0,1,2,3], widths=0.48, patch_artist=True,
    medianprops   = dict(color='#111111', linewidth=2.5),
    boxprops      = dict(linewidth=1.4),
    whiskerprops  = dict(linewidth=1.2, linestyle='--', color='#555555'),
    capprops      = dict(linewidth=1.4, color='#555555'),
    flierprops    = dict(marker=''),
    zorder=3,
)
for patch, col in zip(_bp2['boxes'], _COLORS2):
    patch.set_facecolor(col); patch.set_alpha(0.5); patch.set_edgecolor('#333333')

for xi, (d, col) in enumerate(zip(_data2, _COLORS2)):
    jitter = _rng2.uniform(-0.10, 0.10, _n2)
    ax2.scatter(xi + jitter, d, color=col, s=30, alpha=0.75, zorder=4, linewidths=0)

ax2.axhline(0, color='#999999', linewidth=0.9, linestyle='--', zorder=2)

ax2.set_xticks([0,1,2,3])
ax2.set_xticklabels(_COND_LABELS2, fontsize=13, linespacing=1.35)
ax2.tick_params(axis='x', length=0, pad=8)
ax2.tick_params(axis='y', labelsize=11, length=4)
ax2.set_ylabel('LOO-ISPC', fontsize=14, labelpad=10)
ax2.set_xlim(-0.75, 3.75)
for _sp in ['top', 'right']:
    ax2.spines[_sp].set_visible(False)
for _sp in ['left', 'bottom']:
    ax2.spines[_sp].set_color('#333333')
    ax2.spines[_sp].set_linewidth(1.0)

for label, color in zip(ax2.get_xticklabels(), _COLORS2):
    label.set_color(color)
    label.set_fontweight('bold')

_ya2  = np.concatenate(_data2)
_ymax2, _ymin2 = _ya2.max(), _ya2.min()
_yr2  = _ymax2 - _ymin2

def _sig_stars2(p): return '***' if p < 0.001 else '**' if p < 0.01 else '*'

def _compound_bracket(ax, x0, x1, x2, x3, y_arm, y_bridge, stars_str, p_val, yr):
    col, lw, tk = '#1a1a1a', 1.4, yr * 0.022
    ax.plot([x0, x1], [y_arm, y_arm],    color=col, lw=lw, clip_on=False)
    ax.plot([x0, x0], [y_arm-tk, y_arm], color=col, lw=lw, clip_on=False)
    ax.plot([x1, x1], [y_arm, y_bridge], color=col, lw=lw, clip_on=False)
    ax.plot([x2, x3], [y_arm, y_arm],    color=col, lw=lw, clip_on=False)
    ax.plot([x3, x3], [y_arm-tk, y_arm], color=col, lw=lw, clip_on=False)
    ax.plot([x2, x2], [y_arm, y_bridge], color=col, lw=lw, clip_on=False)
    ax.plot([x1, x2], [y_bridge, y_bridge], color=col, lw=lw, clip_on=False)
    xm = (x1 + x2) / 2
    ax.text(xm, y_bridge + yr*0.015, stars_str,
            ha='center', va='bottom', fontsize=17, color=col, fontweight='bold')
    ax.text(xm, y_bridge + yr*0.1, f'$p_{{fdr}}$ = {p_val:.3f}',
            ha='center', va='bottom', fontsize=10.5, color='#333333')

if _row2['sig_agreement']:
    _compound_bracket(ax2, -0.45, 1.45, 1.55, 3.45,
                      _ymax2 + _yr2*0.10, _ymax2 + _yr2*0.20,
                      _sig_stars2(_row2['p_fdr_agreement']),
                      _row2['p_fdr_agreement'], _yr2)

if _row2['sig_group']:
    _y_og = _ymax2 + _yr2*0.32; _y_ig = _ymax2 + _yr2*0.44
    _tk2 = _yr2*0.022; _col2 = '#1a1a1a'; _lw2 = 1.4
    ax2.plot([0.55,2.45],[_y_og,_y_og], color=_col2,lw=_lw2,clip_on=False)
    ax2.plot([0.55,0.55],[_y_og-_tk2,_y_og],color=_col2,lw=_lw2,clip_on=False)
    ax2.plot([2.45,2.45],[_y_og-_tk2,_y_og],color=_col2,lw=_lw2,clip_on=False)
    ax2.plot([-0.45,3.45],[_y_ig,_y_ig],color=_col2,lw=_lw2,clip_on=False)
    ax2.plot([-0.45,-0.45],[_y_ig-_tk2,_y_ig],color=_col2,lw=_lw2,clip_on=False)
    ax2.plot([3.45,3.45],[_y_ig-_tk2,_y_ig],color=_col2,lw=_lw2,clip_on=False)
    ax2.text(1.5,(_y_og+_y_ig)/2,_sig_stars2(_row2['p_fdr_group']),
             ha='center',va='center',fontsize=17,color=_col2,fontweight='bold')
    ax2.text(1.5, _y_ig+_tk2*0.8, f'$p_{{fdr}}$ = {_row2["p_fdr_group"]:.3f}',
             ha='center',va='bottom',fontsize=10.5,color='#333333')

_n_br2 = sum([_row2['sig_agreement'], _row2['sig_group'], _row2['sig_interaction']])
ax2.set_ylim(_ymin2 - _yr2*0.06, _ymax2 + _yr2*(0.38 + _n_br2*0.15))

plt.tight_layout(pad=1.5)

from matplotlib.transforms import blended_transform_factory as _btf
_bar_trans = _btf(ax2.transData, ax2.transAxes)
for xi, col in enumerate(_COLORS2):
    ax2.plot([xi - 0.24, xi + 0.24], [-0.16, -0.16],
             transform=_bar_trans, color=col, linewidth=5,
             solid_capstyle='butt', clip_on=False)

_out2 = ANOVA_DIR / 'anova_boxplot_4cond_agreement.png'
fig2.savefig(str(_out2), dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved -> {_out2}')

In [ ]:
# ── Interaction lines plot — 7Networks_LH_Vis_21 ────────────────────────────
# Agreement × Group interaction (= Pro vs Anti valence)
# x-axis: Ingroup → Outgroup;  lines: Agreed (green) vs Disagreed (red)

_PARCEL_LV = '7Networks_LH_Vis_21'
_idx_lv    = ALL_PARCEL_NAMES.index(_PARCEL_LV)
_rows_lv   = anova_df[anova_df['parcel_name'] == _PARCEL_LV]
if len(_rows_lv) == 0:
    raise ValueError(f'{_PARCEL_LV} not in ANOVA parcel set.')
_row_lv = _rows_lv.iloc[0]
_n_lv   = len(common_subjects)

_PL_lv = subj_arr['ProLeft'][:,   _idx_lv]   # Agreed   × Ingroup
_AR_lv = subj_arr['AntiRight'][:, _idx_lv]   # Agreed   × Outgroup
_AL_lv = subj_arr['AntiLeft'][:,  _idx_lv]   # Disagreed × Ingroup
_PR_lv = subj_arr['ProRight'][:,  _idx_lv]   # Disagreed × Outgroup

_COL_AG_lv = '#77a650'   # Agreed   — green (matches interaction box_plot_anova)
_COL_DG_lv = '#bb4e31'   # Disagreed — red-orange
_rng_lv    = np.random.default_rng(42)

fig_lv, ax_lv = plt.subplots(figsize=(6, 6))
fig_lv.patch.set_facecolor('white')

# Subject-level thin lines
for pl, ar in zip(_PL_lv, _AR_lv):
    ax_lv.plot([0, 1], [pl, ar], color=_COL_AG_lv, alpha=0.15, linewidth=0.7, zorder=1)
for al, pr in zip(_AL_lv, _PR_lv):
    ax_lv.plot([0, 1], [al, pr], color=_COL_DG_lv, alpha=0.15, linewidth=0.7, zorder=1)

# Scatter dots at each node
for x_pos, vals_ag, vals_dg in [(0, _PL_lv, _AL_lv), (1, _AR_lv, _PR_lv)]:
    j_ag = _rng_lv.uniform(-0.06, 0.06, _n_lv)
    j_dg = _rng_lv.uniform(-0.06, 0.06, _n_lv)
    ax_lv.scatter(x_pos + j_ag, vals_ag, color=_COL_AG_lv, s=24, alpha=0.65, zorder=3, linewidths=0)
    ax_lv.scatter(x_pos + j_dg, vals_dg, color=_COL_DG_lv, s=24, alpha=0.65, zorder=3, linewidths=0)

# Group mean lines
ax_lv.plot([0, 1], [_PL_lv.mean(), _AR_lv.mean()], 'o-', color=_COL_AG_lv,
           linewidth=3.0, markersize=9, zorder=5, label='Agreed')
ax_lv.plot([0, 1], [_AL_lv.mean(), _PR_lv.mean()], 'o-', color=_COL_DG_lv,
           linewidth=3.0, markersize=9, zorder=5, label='Disagreed')

ax_lv.axhline(0, color='#999999', linewidth=0.9, linestyle='--', zorder=0)

ax_lv.set_xticks([0, 1])
ax_lv.set_xticklabels(['Ingroup', 'Outgroup'], fontsize=13)
ax_lv.tick_params(axis='x', length=0, pad=8)
ax_lv.tick_params(axis='y', labelsize=11, length=4)
ax_lv.set_ylabel('LOO-ISPC', fontsize=14, labelpad=10)
ax_lv.set_xlim(-0.35, 1.35)
for _sp in ['top', 'right']:
    ax_lv.spines[_sp].set_visible(False)
for _sp in ['left', 'bottom']:
    ax_lv.spines[_sp].set_color('#333333')
    ax_lv.spines[_sp].set_linewidth(1.0)

ax_lv.legend(fontsize=11, loc='lower right', frameon=False)

# Significance bracket for interaction
_ya_lv   = np.concatenate([_PL_lv, _AR_lv, _AL_lv, _PR_lv])
_ymax_lv = _ya_lv.max()
_ymin_lv = _ya_lv.min()
_yr_lv   = _ymax_lv - _ymin_lv
_col_lv  = '#1a1a1a'
_lw_lv   = 1.4

def _stars_lv(p): return '***' if p < 0.001 else '**' if p < 0.01 else '*'

if _row_lv['sig_interaction']:
    _tk_lv = _yr_lv * 0.025
    _y_br  = _ymax_lv + _yr_lv * 0.10
    ax_lv.plot([-0.05, -0.05, 1.05, 1.05],
               [_y_br, _y_br + _tk_lv, _y_br + _tk_lv, _y_br],
               color=_col_lv, lw=_lw_lv, clip_on=False)
    ax_lv.text(0.5, _y_br + _tk_lv * 2.0,
               _stars_lv(_row_lv['p_fdr_interaction']),
               ha='center', va='bottom', fontsize=17, color=_col_lv, fontweight='bold')
    ax_lv.text(0.5, _y_br + _tk_lv * 2.0 + _yr_lv * 0.09,
               f'$p_{{fdr}}$ = {_row_lv["p_fdr_interaction"]:.3f}',
               ha='center', va='bottom', fontsize=10.5, color='#333333')

_y_top_lv = _ymax_lv + _yr_lv * (0.42 if _row_lv['sig_interaction'] else 0.12)
ax_lv.set_ylim(_ymin_lv - _yr_lv * 0.06, _y_top_lv)

plt.tight_layout(pad=1.5)

_out_lv = ANOVA_DIR / 'interaction_lines_LH_Vis_21.png'
fig_lv.savefig(str(_out_lv), dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved -> {_out_lv}')